# Program 7: RNN and LSTM Character-Level Language Model
## Objective: Train a character-level language model to generate new text

## Task 1: Load Text and Build Character-to-Index Mappings

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

# Load text data
text_data = """
Roses are red,
Violets are blue,
Sugar is sweet,
And so are you.

Twinkle, twinkle, little star,
How I wonder what you are.
Up above the world so high,
Like a diamond in the sky.

There once was a young man from Peru,
Who dreamed he was eating his shoe.
He awoke in a fright,
In the middle of the night,
To find that his dream had come true.
"""

# Create character vocabulary
unique_chars = sorted(set(text_data))
vocab_size = len(unique_chars)

# Create char-to-index and index-to-char mappings
char_to_idx = {char: idx for idx, char in enumerate(unique_chars)}
idx_to_char = {idx: char for idx, char in enumerate(unique_chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Unique characters: {unique_chars}")
print(f"\nChar-to-Index mapping (first 10): {dict(list(char_to_idx.items())[:10])}")
print(f"Index-to-Char mapping (first 10): {dict(list(idx_to_char.items())[:10])}")

Vocabulary size: 37
Unique characters: ['\n', ' ', ',', '.', 'A', 'H', 'I', 'L', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']

Char-to-Index mapping (first 10): {'\n': 0, ' ': 1, ',': 2, '.': 3, 'A': 4, 'H': 5, 'I': 6, 'L': 7, 'P': 8, 'R': 9}
Index-to-Char mapping (first 10): {0: '\n', 1: ' ', 2: ',', 3: '.', 4: 'A', 5: 'H', 6: 'I', 7: 'L', 8: 'P', 9: 'R'}


## Task 2: Vectorize Text and Create Character-Index Mapping

In [2]:
# Vectorize the text: convert characters to indices
text_as_int = np.array([char_to_idx[char] for char in text_data])

print(f"Text length: {len(text_data)} characters")
print(f"Text as integers (first 50): {text_as_int[:50]}")
print(f"Sample: text[0:10] = '{text_data[0:10]}' -> indices = {text_as_int[0:10]}")

Text length: 343 characters
Text as integers (first 50): [ 0  9 28 31 19 31  1 15 30 19  1 30 19 18  2  0 13 23 28 25 19 32 31  1
 15 30 19  1 16 25 33 19  2  0 10 33 21 15 30  1 23 31  1 31 35 19 19 32
  2  0]
Sample: text[0:10] = '
Roses are' -> indices = [ 0  9 28 31 19 31  1 15 30 19]


## Task 3: Create Training Sequences (40 chars input → 41st char output)

In [3]:
# Create sequences: input of 40 chars and target of 1 char (41st)
seq_length = 40

X_train = []
y_train = []

for i in range(len(text_as_int) - seq_length):
    X_train.append(text_as_int[i:i + seq_length])
    y_train.append(text_as_int[i + seq_length])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Training samples: {len(X_train)}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nSample sequence:")
print(f"Input (40 chars): {X_train[0]} -> '{[idx_to_char[idx] for idx in X_train[0]]}'")
print(f"Target (1 char): {y_train[0]} -> '{idx_to_char[y_train[0]]}'")

Training samples: 303
X_train shape: (303, 40)
y_train shape: (303,)

Sample sequence:
Input (40 chars): [ 0  9 28 31 19 31  1 15 30 19  1 30 19 18  2  0 13 23 28 25 19 32 31  1
 15 30 19  1 16 25 33 19  2  0 10 33 21 15 30  1] -> '['\n', 'R', 'o', 's', 'e', 's', ' ', 'a', 'r', 'e', ' ', 'r', 'e', 'd', ',', '\n', 'V', 'i', 'o', 'l', 'e', 't', 's', ' ', 'a', 'r', 'e', ' ', 'b', 'l', 'u', 'e', ',', '\n', 'S', 'u', 'g', 'a', 'r', ' ']'
Target (1 char): 23 -> 'i'


## Task 4: Build RNN Model with SimpleRNN Layers

In [4]:
# Build RNN Model
model = keras.Sequential([
    layers.Embedding(vocab_size, 64, input_length=seq_length),
    layers.SimpleRNN(256, return_sequences=True),
    layers.SimpleRNN(256),
    layers.Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/Users/sumith/Desktop/5 sem/tfenv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-11-21 10:45:16.254474: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-21 10:45:16.254508: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-21 10:45:16.254510: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-21 10:45:16.254542: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-21 10:45:16.254554: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)

/Users/sumith/Desktop/5 sem/tfenv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-11-21 10:45:16.254474: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-21 10:45:16.254508: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-21 10:45:16.254510: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-21 10:45:16.254542: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-21 10:45:16.254554: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Task 5: Train the Model

In [5]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

print(f"\nTraining completed!")
print(f"Final loss: {history.history['loss'][-1]:.4f}")
print(f"Final accuracy: {history.history['accuracy'][-1]:.4f}")

Epoch 1/50


Epoch 1/50


2025-11-21 10:45:16.954578: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Epoch 1/50


2025-11-21 10:45:16.954578: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.0858 - loss: 3.4707
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.0858 - loss: 3.4707
Epoch 2/50
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.1485 - loss: 3.2394
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.1485 - loss: 3.2394
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2013 - loss: 2.9553
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2013 - loss: 2.9553
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.1914 - loss: 2.9972
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.1914 - loss: 2.9972
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2805 - loss: 2.6240
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2805 - loss: 2.6240
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.3762 - loss: 2.2359
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.3762 - loss: 2.2359
Epo

## Task 6: Write Text Generation Function

In [8]:
def generate_text(seed_text, num_chars=100, temperature=1.0):
    """
    Generate text using the trained model
    
    Args:
        seed_text: Initial text to start generation
        num_chars: Number of characters to generate
        temperature: Controls randomness (0.1=deterministic, 1.0=normal, >1=random)
    
    Returns:
        Generated text string
    """
    generated = seed_text
    
    for _ in range(num_chars):
        # Get last 40 characters
        input_seq = generated[-seq_length:]
        
        # Convert to indices
        input_indices = np.array([char_to_idx[char] for char in input_seq])
        input_indices = input_indices.reshape(1, -1)
        
        # Predict next character
        predictions = model.predict(input_indices, verbose=0)
        
        # Apply temperature
        predictions = np.log(predictions + 1e-10) / temperature
        predictions = np.exp(predictions) / np.sum(np.exp(predictions))
        
        # Sample from distribution
        next_idx = np.random.choice(vocab_size, p=predictions[0])
        next_char = idx_to_char[next_idx]
        
        generated += next_char
    
    return generated

print("Text generation function created!")

Text generation function created!


## Task 7: Generate New Text and Validate

In [9]:
# Test text generation with different seed texts
print("=" * 80)
print("GENERATED TEXT SAMPLES")
print("=" * 80)

seed_texts = [
    "Roses",
    "Twinkle",
    "There once"
]

for seed in seed_texts:
    print(f"\nSeed: '{seed}'")
    print("-" * 40)
    generated = generate_text(seed, num_chars=80, temperature=0.7)
    print(generated)
    print()

GENERATED TEXT SAMPLES

Seed: 'Roses'
----------------------------------------
RosessgwThe ead nreoi d ar dree had h ois  aridhtei ng me s aru 
o   d mi hr  ohae tn


Seed: 'Twinkle'
----------------------------------------
RosessgwThe ead nreoi d ar dree had h ois  aridhtei ng me s aru 
o   d mi hr  ohae tn


Seed: 'Twinkle'
----------------------------------------
Twinkle thi  are foh.
 owsitrke he talonser ereesthe  de s  rn hoe igiam
no tame w l,d 


Seed: 'There once'
----------------------------------------
Twinkle thi  are foh.
 owsitrke he talonser ereesthe  de s  rn hoe igiam
no tame w l,d 


Seed: 'There once'
----------------------------------------
There once   a a tos eihli iros  tre,
Wh tdreameaa eow g sisIemthie awre.
He a hed he  in 

There once   a a tos eihli iros  tre,
Wh tdreameaa eow g sisIemthie awre.
He a hed he  in 

